# 06 — A100 router training (DeBERTa)

Fine-tunes the multitask query router on the leak-free `data/router_large` set.

Heads match `src/query_analyzer.py`: retrieval necessity (2-way), temporal class (`current` / `historical` / `timeless`), and the confidence head. The confidence term is binary cross-entropy on detached prediction-correctness times the row's `label_confidence`. It is applied outside `forward`, so DistilBERT training in `train_analyzer` is unchanged.

This notebook writes a **new** bundle (`analyzer.pt` + `analyzer_meta.json`) to Google Drive. It does not replace `artifacts/analyzer.pt`.

Runtime: Colab Pro **A100**. Budget for the 6-trial sweep plus one final fit is about 2–3 hours. Set `USE_LARGE = True` only if you accept a smaller batch and gradient checkpointing for `microsoft/deberta-v3-large`.

In [ ]:
import os
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '180'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '30'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip install -q "transformers>=4.41" "accelerate>=0.27" sentencepiece protobuf optuna


In [ ]:
import sys
import zipfile
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/adaptive-temporal-rag/router_large_colab_bundle.zip'
    OUT_DIR = Path('/content/drive/MyDrive/adaptive-temporal-rag/router_large_a100')
    ROOT = Path('/content/router_bundle')
    if not (ROOT / 'src' / 'router_large.py').exists():
        with zipfile.ZipFile(ZIP_PATH) as bundle:
            bundle.extractall(ROOT)
else:
    ROOT = Path('..').resolve()
    if not (ROOT / 'src').exists():
        ROOT = Path('.').resolve()
    OUT_DIR = ROOT / 'artifacts' / 'router_large_a100'

OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
print('ROOT', ROOT)
print('OUT_DIR', OUT_DIR)
print('colab', IN_COLAB)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime. This notebook is meant for an A100.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU', gpu_name)
print('bf16', torch.cuda.is_bf16_supported())
if 'a100' not in gpu_name.lower() and 'h100' not in gpu_name.lower():
    print('WARNING: sized for an A100. On this GPU the trainer keeps fp32 when bf16 is unsupported.')


## Config

`RUN_SWEEP` runs Optuna for 6 trials over learning rate `{5e-6, 1e-5, 2e-5}`, batch `{16, 32}` (or `{4, 8}` for the large model), and max epochs `{3, 5}`. If Optuna is not importable, the same budget uses a fixed 6-point grid. Each trial early-stops on validation macro-F1 (patience 1). The winning hyperparameters are retrained once with patience 2.

Training uses class-weighted cross-entropy, cosine decay with warmup, gradient accumulation, bf16 autocast on A100, and `torch.compile` when it survives a first-step probe. The large model turns compile off and uses gradient checkpointing instead.

In [ ]:
USE_LARGE = False
RUN_SWEEP = True
N_TRIALS = 6

MODEL_NAME = 'microsoft/deberta-v3-large' if USE_LARGE else 'microsoft/deberta-v3-base'
BATCH_CHOICES = (4, 8) if USE_LARGE else (16, 32)
GRAD_ACCUM = 4 if USE_LARGE else 1
MAX_LENGTH = 128
GRADIENT_CHECKPOINTING = bool(USE_LARGE)
PATIENCE = 2
CONFIDENCE_LOSS_WEIGHT = 0.3
print(MODEL_NAME, 'batch choices', BATCH_CHOICES, 'accum', GRAD_ACCUM)


In [ ]:
import json
from src.router_large import load_split_rows, verify_saved_leakage

data_dir = ROOT / 'data' / 'router_large'
stats = json.loads((data_dir / 'stats.json').read_text(encoding='utf-8'))
train_rows = load_split_rows(data_dir / 'train.jsonl')
val_rows = load_split_rows(data_dir / 'val.jsonl')
test_rows = load_split_rows(data_dir / 'test.jsonl')
print('splits', len(train_rows), len(val_rows), len(test_rows), 'total', stats['total'])
print('temporal', stats['by_temporal_class'])
print('needs_retrieval', stats['by_needs_retrieval'])
print('by_source', stats['by_source'])
print('synthetic_fraction', round(stats['synthetic_fraction'], 4))
print('constraints', stats['constraints'])
print('failed sources', [item['source'] for item in stats.get('sources_failed', [])])
for name, rows in (('train', train_rows), ('val', val_rows), ('test', test_rows)):
    report = verify_saved_leakage(stats, rows)
    print(name, 'residual leaks', report['leaks'])
    if report['leaks']:
        raise RuntimeError(f'{name} overlaps a forbidden test question, entity, or template')


In [ ]:
from src.router_large import sweep_hyperparams

best = {'lr': 1e-5, 'batch_size': BATCH_CHOICES[-1], 'max_epochs': 4}
if RUN_SWEEP:
    sweep = sweep_hyperparams(
        train_rows,
        val_rows,
        model_name=MODEL_NAME,
        n_trials=N_TRIALS,
        use_optuna=True,
        batch_choices=BATCH_CHOICES,
        grad_accum=GRAD_ACCUM,
        max_length=MAX_LENGTH,
        patience=1,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        confidence_loss_weight=CONFIDENCE_LOSS_WEIGHT,
        use_compile=not GRADIENT_CHECKPOINTING,
        use_bf16=True,
    )
    print('sweep backend', sweep['backend'])
    print('best trial', sweep['best'])
    (OUT_DIR / 'sweep.json').write_text(json.dumps(sweep, indent=2, default=str), encoding='utf-8')
    if sweep['best']:
        best = sweep['best']
else:
    print('sweep skipped; using', best)


In [ ]:
from src.router_large import train_large_router

result = train_large_router(
    train_rows,
    val_rows,
    output_dir=OUT_DIR,
    model_name=MODEL_NAME,
    epochs=int(best['max_epochs']),
    batch_size=int(best['batch_size']),
    lr=float(best['lr']),
    max_length=MAX_LENGTH,
    grad_accum=GRAD_ACCUM,
    patience=PATIENCE,
    confidence_loss_weight=CONFIDENCE_LOSS_WEIGHT,
    use_compile=not GRADIENT_CHECKPOINTING,
    use_bf16=True,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    fit_temperature=True,
)
print('best_val_macro_f1', result['best_val_macro_f1'])
print('temperature', result['temperature'])
print('precision', result['precision'], 'compiled', result['compiled'])
print('saved', OUT_DIR / 'analyzer.pt')


## Evaluation

Reports macro-F1, per-class F1, confusion matrices (rows are gold, columns are predictions, order `current`, `historical`, `timeless`), and 15-bin ECE.

- `router_large_test` is the leak-free split built with the training set.
- `heldout_test` and `timeqa_test` are the existing external sets. They were excluded from training, so overlap with them is expected and is not a leakage failure.

In [ ]:
from src.data_schema import load_queries
from src.router_large import evaluate_rows, load_large_router

service = load_large_router(OUT_DIR)

def run_eval(name, rows):
    metrics = evaluate_rows(
        service.model,
        service.tokenizer,
        rows,
        service.device,
        max_length=service.max_length,
        temperature=service.temperature,
        batch_size=64,
    )
    metrics.pop('temporal_logits', None)
    metrics.pop('temporal_labels', None)
    print('\n==', name, 'n=', metrics['n'])
    print('macro_f1', round(metrics['val_macro_f1'], 4))
    print('temporal_macro_f1', round(metrics['temporal']['macro_f1'], 4), 'per class', metrics['temporal']['per_class'])
    print('necessity_macro_f1', round(metrics['necessity']['macro_f1'], 4), 'per class', metrics['necessity']['per_class'])
    print('temporal_confusion', metrics['temporal_confusion'])
    print('necessity_confusion', metrics['necessity_confusion'])
    print('ece temporal', round(metrics['temporal_ece'], 4), 'necessity', round(metrics['necessity_ece'], 4), 'confidence head', round(metrics.get('confidence_head_ece', float('nan')), 4))
    return metrics

reports = {'router_large_test': run_eval('leak-free test', test_rows)}
held_path = ROOT / 'data' / 'heldout' / 'queries.jsonl'
time_path = ROOT / 'data' / 'general' / 'timeqa' / 'queries.jsonl'
if held_path.exists():
    held_rows = [row for row in load_queries(held_path) if row.split == 'test']
    reports['heldout_test'] = run_eval('heldout test', held_rows)
else:
    print('missing held-out queries', held_path)
if time_path.exists():
    time_rows = [row for row in load_queries(time_path) if row.split == 'test']
    reports['timeqa_test'] = run_eval('timeqa test', time_rows)
else:
    print('missing TimeQA queries', time_path)

(OUT_DIR / 'eval_report.json').write_text(json.dumps(reports, indent=2), encoding='utf-8')
print('bundle', OUT_DIR / 'analyzer.pt')
print('meta', OUT_DIR / 'analyzer_meta.json')
print('Load later with: from src.router_large import load_large_router')
